# Keep-only controls - matched capacity, and relation identity vs. relation count

`ablation_keep_disease_only.ipynb` found that reducing the KG to **9 edge types of 50** (the three
drug-disease task relations plus the two the similarity signature reads) costs only
**-0.0658 indication / -0.0506 contraindication**, retaining ~80% of the above-chance margin. Two
objections to that result are left open, and this notebook closes both. It does not modify that
notebook - it imports its results as references.

| arm | relations kept | edge types | encoder params | new? |
|---|---|---|---|---|
| `full_kg` | all | 50 | 1,010,000 | imported |
| `only_disease_relations` | task + `disease_disease` + `disease_protein` | 9 | 181,800 | imported |
| `only_disease_matched` | same 9, encoder widened to matched capacity | 9 | **1,010,880** | **yes** |
| `dd_plus_padding` | task + `disease_disease` + inert padding | 9 | 181,800 | **yes** |
| `dp_plus_padding` | task + `disease_protein` + inert padding | 9 | 181,800 | **yes** |

## Objection 1 - the keep-only arm also lost 82% of its encoder

`HeteroRGCNLayer` holds one `nn.Linear` per relation, so cutting 50 edge types to 9 cuts the encoder
from 1,010,000 to 181,800 parameters. The encoder-swap grid showed this metric is very
capacity-sensitive, so part of the -0.066 is lost capacity and **-0.066 is an upper bound** on what the
41 dropped relations carry, not an estimate.

`only_disease_matched` restores the capacity. With 9 edge types, `n_hid = n_out = 288` and `n_inp`
**left at 100** gives

    2 layers, per relation: (100*288 + 288) + (288*288 + 288) = 112,320
    x 9 relations                                             = 1,010,880

within 0.09% of the baseline's 1,010,000. Section 3 asserts this against the real model rather than
trusting the arithmetic.

Two things to be honest about, both logged per run:

- **`n_inp` stays at 100 on purpose.** `initialize_node_embedding` sizes the frozen random node features
  from `n_inp`, so widening it would hand this arm richer input features as well as a bigger encoder -
  a second, uncontrolled advantage. Keeping it at 100 means the input representation is identical to
  every other arm in the study and only the encoder grows.
- **This matches *encoder* capacity, not total parameters.** `n_hid` also sizes the `rarity` gate
  (`nn.Linear(n_hid*2, 1)`) and `n_out` sizes `w_rels` (`(n_etypes, n_out)`), so the decoder changes a
  little too: `w_rels` goes from 50x100 = 5,000 to 9x288 = 2,592 and each gate from 201 to 577
  parameters. Net non-encoder difference is ~1,900 parameters against an encoder of a million, so it
  cannot carry the result, but the notebook records `n_params` for both and the write-up should say
  "matched encoder capacity".

## Objection 2 - is it those two relations, or just having nine edge types?

`only_disease_relations` keeps the two relations the signature reads. A sceptic can say the result is
about **count** - any 9 edge types would do - rather than **identity**.

### Why the obvious control cannot be run

The version of this control written into `docs/ABLATION_FINDINGS.md` (section 11) was "task labels plus
three arbitrary non-disease relations of comparable size". That arm is **predictably degenerate** and
would waste 20 runs. `complex_disease` is zero-shot, so a held-out test disease has no training
drug-disease edges; if neither `disease_disease` nor `disease_protein` is kept, such a disease has **no
edges at all** (zero embedding) *and* an all-zero signature (zero prototype), so every DistMult score is
0 and AUPRC is exactly 0.500 on every seed. That is the `dd_only` mechanism already documented in
section 7 of the findings doc. `INCLUDE_DEGENERATE_CONTROL` below runs it anyway on a couple of seeds if
you want the prediction confirmed - **not** on 20.

### What is run instead

Hold the edge-type count at 9 and the encoder at 181,800 - identical to `only_disease_relations` - and
keep **exactly one** of the two signature relations, padding the rest of the budget with relations that
are *deliberately inert* for the task:

| arm | disease-connecting relation | padding | edge types |
|---|---|---|---|
| `dd_plus_padding` | `disease_disease` (1 etype, 64,388 edges) | `molfunc_protein` (2 etypes, 139,060 edges) | 9 |
| `dp_plus_padding` | `disease_protein` (2 etypes, 160,822 edges) | `phenotype_phenotype` (1 etype, 37,472 edges) | 9 |

The padding is inert because it forms a component disjoint from the drug-disease task: in
`dd_plus_padding` nothing links `gene/protein` to `disease`, so the `gene/protein`-`molecular_function`
edges cannot reach a prediction; in `dp_plus_padding` `effect/phenotype` is isolated. The padding exists
only to hold the edge-type count, and therefore the encoder size, constant. The relation sizes were
also chosen to roughly match what they stand in for (`molfunc_protein` 139,060 vs `disease_protein`
160,822; `phenotype_phenotype` 37,472 vs `disease_disease` 64,388) so that pretraining sees a comparable
number of edges.

This is a strictly better control than the arbitrary one: at **fixed** edge-type count and **fixed**
encoder size it decomposes `only_disease_relations` into its two halves, and `only_disease_relations`
minus each arm is the marginal value of the other relation with capacity held constant - which
leave-one-out from the full KG cannot give, because there capacity moves too.

---
## 1 - Environment

In [ ]:
import os, sys, json, time, gc, random, shutil, inspect, platform, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')


def _find_repo_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / 'pyg_implementation' / 'txgnn').is_dir():
            return cand
    raise RuntimeError('Could not find pyg_implementation/txgnn above ' + str(p))


REPO_ROOT = _find_repo_root()
PYG_ROOT  = REPO_ROOT / 'pyg_implementation'
sys.path.insert(0, str(PYG_ROOT))   # PyG port FIRST, so `import txgnn` never hits dgl_implementation/

import torch
import torch.nn as nn

import txgnn
from txgnn import TxData, TxGNN
from txgnn.utils import evaluate_fb, create_pyg_graph, get_n_params

assert 'pyg_implementation' in txgnn.__file__, (
    'Wrong txgnn package on sys.path (%s). Restart the kernel.' % txgnn.__file__)

_MI_PARAMS = set(inspect.signature(TxGNN.model_initialize).parameters)
assert 'use_decision_network' not in _MI_PARAMS, (
    'model_initialize accepts use_decision_network again - revisit this notebook.')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


print('repo  :', REPO_ROOT)
print('txgnn :', txgnn.__file__)
print('torch :', torch.__version__, '| device:', DEVICE,
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else platform.processor())

---
## 2 - Configuration

`MODEL_CFG` is the base config, byte-identical to every other grid. Only `only_disease_matched`
overrides it, and only in `n_hid` / `n_out`. `SEEDS` is the same 20-seed set, so both reference arms
import free from `ablation_keep_disease_only`.

In [ ]:
SPLIT   = 'complex_disease'
SEEDS   = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 31, 41, 42, 63, 70, 77, 86, 89, 95, 98]
PROFILE = 'full'

MODEL_CFG = dict(
    n_hid       = 100,
    n_inp       = 100,
    n_out       = 100,
    proto       = True,
    proto_num   = 3,
    attention   = False,
    sim_measure = 'all_nodes_profile',
    agg_measure = 'rarity',
)

# matched-capacity width for a 9-edge-type encoder; n_inp deliberately stays at 100 (see cell 0)
MATCHED_WIDTH = 288

PRETRAIN_CFG = dict(n_epoch=1,    learning_rate=1e-3, batch_size=1024, train_print_per_n=500)
FINETUNE_CFG = dict(n_epoch=1000, learning_rate=5e-4, train_print_per_n=100, valid_per_n=20)

DD_REL     = ['contraindication', 'indication', 'off-label use']
DD_REV_REL = ['rev_' + r for r in DD_REL]
DD_ETYPES  = [('drug', r, 'disease') for r in DD_REL] + [('disease', r, 'drug') for r in DD_REV_REL]

SIG_ETYPES = ['disease_disease', 'rev_disease_protein']   # library default, model.py:88
SIG_NODES  = ['disease', 'gene/protein']

DATA_FOLDER = REPO_ROOT / 'data'
ABL_ROOT    = REPO_ROOT / 'results' / 'ablation_keep_only_controls'
KEEP_ROOT   = REPO_ROOT / 'results' / 'ablation_keep_disease_only'   # read-only: both references
ABL_ROOT.mkdir(parents=True, exist_ok=True)

REF_ARM     = 'full_kg'                  # primary reference for every delta
REF_ARM_2   = 'only_disease_relations'   # secondary reference, for the capacity + identity contrasts

# --- switches ------------------------------------------------------------------------------------
RUN_GRID          = True
SAVE_MODELS       = False
IMPORT_REFS       = True   # pull full_kg and only_disease_relations from ablation_keep_disease_only
INCLUDE_DEGENERATE_CONTROL = False  # the predictably-0.500 arm; see cell 0. Use DEGENERATE_SEEDS only.
DEGENERATE_SEEDS  = [1, 2]
STOP_AFTER_N_RUNS = None

print('split  :', SPLIT)
print('seeds  : %d -> %s' % (len(SEEDS), SEEDS))
print('results:', ABL_ROOT.relative_to(REPO_ROOT))

---
## 3 - Relations, arms, and the parameter arithmetic

`expected_encoder_params` is the closed form for a 2-layer encoder; `run_one` asserts the real model
against it, computed from the edge-type count of the graph actually built. A "matched capacity" arm that
is not actually matched therefore fails loudly instead of quietly producing a number.

In [ ]:
DD_RELATIONS        = ['indication', 'contraindication', 'off-label use']
SIGNATURE_RELATIONS = ['disease_disease', 'disease_protein']

# inert padding: chosen to hold the edge-type count at 9 while contributing no disease-reachable path.
# sizes are matched to what they stand in for (see cell 0).
PAD_FOR_DP = 'molfunc_protein'      # 2 etypes, 139,060 edges  <- stands in for disease_protein
PAD_FOR_DD = 'phenotype_phenotype'  # 1 etype,   37,472 edges  <- stands in for disease_disease

# relations that contribute 1 edge type (same source/target node type) rather than 2
SAME_TYPE_RELATIONS = {
    'disease_disease', 'drug_drug', 'protein_protein', 'phenotype_phenotype', 'anatomy_anatomy',
    'molfunc_molfunc', 'cellcomp_cellcomp', 'pathway_pathway', 'bioprocess_bioprocess',
    'exposure_exposure',
}


def n_etypes_for(keep):
    '''Edge types a `keep` list produces: 1 for same-type relations, 2 otherwise.'''
    return sum(1 if r in SAME_TYPE_RELATIONS else 2 for r in keep)


def expected_encoder_params(n_etypes, n_inp, n_hid, n_out):
    return n_etypes * ((n_inp * n_hid + n_hid) + (n_hid * n_out + n_out))


def _arm(name, keep, overrides=None, note='', seeds=None):
    cfg = dict(MODEL_CFG, **(overrides or {}))
    n_et = 50 if keep is None else n_etypes_for(keep)
    return {'arm': name, 'keep': keep, 'overrides': overrides or {}, 'note': note,
            'expect_etypes': n_et, 'seeds': seeds,
            'expect_encoder_params': expected_encoder_params(
                n_et, cfg['n_inp'], cfg['n_hid'], cfg['n_out'])}


ARMS = [
    _arm('full_kg', None, note='reference: the unmodified KG (imported)'),
    _arm('only_disease_relations', DD_RELATIONS + SIGNATURE_RELATIONS,
         note='reference: the keep-only result (imported)'),
    _arm('only_disease_matched', DD_RELATIONS + SIGNATURE_RELATIONS,
         overrides={'n_hid': MATCHED_WIDTH, 'n_out': MATCHED_WIDTH},
         note='objection 1: same 9 edge types, encoder capacity restored'),
    _arm('dd_plus_padding', DD_RELATIONS + ['disease_disease', PAD_FOR_DP],
         note='objection 2: disease_disease only, padded to 9 edge types'),
    _arm('dp_plus_padding', DD_RELATIONS + ['disease_protein', PAD_FOR_DD],
         note='objection 2: disease_protein only, padded to 9 edge types'),
]
if INCLUDE_DEGENERATE_CONTROL:
    ARMS.append(_arm('arbitrary9_no_disease', DD_RELATIONS + [PAD_FOR_DP, PAD_FOR_DD],
                     note='PREDICTED DEGENERATE (exactly 0.500) - confirmation only',
                     seeds=DEGENERATE_SEEDS))

ARMS_BY_NAME = {a['arm']: a for a in ARMS}
assert len(ARMS_BY_NAME) == len(ARMS), 'duplicate arm name'
for _r in (REF_ARM, REF_ARM_2):
    assert _r in ARMS_BY_NAME, 'reference arm %r missing' % _r

# every arm except full_kg and the matched one must be 9 edge types at 181,800 params, or the
# "identity vs count" comparison is not at fixed capacity
_BASE_9 = expected_encoder_params(9, 100, 100, 100)
for a in ARMS:
    if a['arm'] in ('full_kg', 'only_disease_matched'):
        continue
    assert a['expect_etypes'] == 9, '%s has %d edge types, expected 9' % (a['arm'], a['expect_etypes'])
    assert a['expect_encoder_params'] == _BASE_9, '%s params %d != %d' % (
        a['arm'], a['expect_encoder_params'], _BASE_9)
_matched = ARMS_BY_NAME['only_disease_matched']['expect_encoder_params']
_full    = expected_encoder_params(50, 100, 100, 100)
assert abs(_matched - _full) / _full < 0.01, 'matched arm is %d vs %d - not within 1%%' % (_matched, _full)

print('%-24s %-11s %-9s %-16s %s' % ('arm', 'edge types', 'width', 'encoder params', '% of full_kg'))
for a in ARMS:
    cfg = dict(MODEL_CFG, **a['overrides'])
    print('%-24s %-11d %-9s %-16s %6.2f%%'
          % (a['arm'], a['expect_etypes'],
             '%d/%d/%d' % (cfg['n_inp'], cfg['n_hid'], cfg['n_out']),
             format(a['expect_encoder_params'], ','),
             100 * a['expect_encoder_params'] / _full))
    print('%-24s %s' % ('', a['note']))

_todo = [a for a in ARMS if a['arm'] not in (REF_ARM, REF_ARM_2)]
print('\n%d new arms x %d seeds = %d runs needing GPU time (2 reference arms import free)'
      % (len(_todo), len(SEEDS), sum(len(a['seeds'] or SEEDS) for a in _todo)))

---
## 4 - Mechanics

`KeptData` mirrors `ablation_keep_disease_only`: `base['df']` is the **untouched** full KG, and
`create_pyg_graph(df_train, df)` reads node counts from it, so every node type survives with identical
indices in every arm and only edges disappear.

`signature_probe` reads back the signatures `DistMultPredictor` actually built. This is the guard that
makes the two padding arms interpretable: each must report exactly **one** of the two signature sources
present and a non-zero maximum. An all-zero signature would mean the arm is degenerate (section 0) and
its number meaningless.

In [ ]:
def expand_relations(names):
    '''Relation base names -> the names present in the frames, incl. rev_ twins.'''
    out = set()
    for r in names:
        out |= {r, 'rev_' + r}
    return out


def keep_relations(df, keep):
    if keep is None:
        return df.reset_index(drop=True)
    return df[df.relation.isin(expand_relations(keep))].reset_index(drop=True)


class KeptData:
    '''Minimal TxData stand-in holding the reduced splits and a rebuilt graph.'''

    def __init__(self, base, keep, data_folder, split):
        self.df = base['df']                      # untouched: fixes node counts across arms
        self.df_train = keep_relations(base['df_train'], keep)
        self.df_valid = keep_relations(base['df_valid'], keep)
        self.df_test  = keep_relations(base['df_test'],  keep)
        self.G = create_pyg_graph(self.df_train, self.df)
        self.data_folder = str(data_folder)
        self.split = split
        self.disease_eval_idx = None
        self.no_kg = False


def encoder_params(model):
    return (sum(p.numel() for p in model.layer1.parameters())
            + sum(p.numel() for p in model.layer2.parameters()))


def signature_probe(model):
    '''(dim, max) of the disease signatures actually built. max == 0 => every signature is zero.'''
    try:
        prof = model.pred.diseases_profile_etypes
        et = next(iter(prof))
        vals = list(prof[et].values())[:64]
        if not vals:
            return None, None
        return int(vals[0].shape[0]), float(max(float(v.max()) for v in vals))
    except Exception:
        return None, None


_SEED_CACHE = {}


def get_seed_data(seed):
    '''Load (and cache) the unablated split frames for one seed. Holds one seed at a time.'''
    if seed in _SEED_CACHE:
        return _SEED_CACHE[seed]
    _SEED_CACHE.clear()
    gc.collect()
    print('[data] loading split seed %d ...' % seed)
    t0 = time.time()
    set_all_seeds(seed)
    d = TxData(data_folder_path=str(DATA_FOLDER))
    d.prepare_split(split=SPLIT, seed=seed, no_kg=False)
    base = {'df': d.df, 'df_train': d.df_train, 'df_valid': d.df_valid, 'df_test': d.df_test}
    del d
    gc.collect()
    _SEED_CACHE[seed] = base
    print('[data] seed %d ready in %.1f s | train edges %d'
          % (seed, time.time() - t0, len(base['df_train'])))
    return base


def run_dir(arm, seed):
    return ABL_ROOT / ('%s_seed%d' % (arm, seed))


def is_done(arm, seed):
    d = run_dir(arm, seed)
    return (d / 'edge_level_test_metrics.csv').exists() and (d / 'run_meta.json').exists()


def arm_seeds(a):
    return a['seeds'] or SEEDS

In [ ]:
# --- importing the two reference arms -------------------------------------------------------------
MODEL_KEYS_CHECKED = sorted(MODEL_CFG)


def import_matching_run(src, dst, expect, extra=None):
    '''Copy a finished run from another grid iff its metadata matches `expect`. Returns (ok, why).'''
    sm, se = src / 'run_meta.json', src / 'edge_level_test_metrics.csv'
    if not (sm.exists() and se.exists()):
        return False, 'no run at %s' % src.name
    meta = json.loads(sm.read_text())
    for k, want in expect.items():
        got = meta.get(k, '<missing>')
        if isinstance(want, dict):
            if not isinstance(got, dict):
                return False, '%s is not a dict' % k
            bad = {kk: (got.get(kk, '<missing>'), vv) for kk, vv in want.items()
                   if got.get(kk, '<missing>') != vv}
            if bad:
                return False, '%s mismatch %s' % (k, bad)
        elif got != want:
            return False, '%s mismatch: %r != %r' % (k, got, want)

    dst.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(se, dst / 'edge_level_test_metrics.csv')
    meta.update(extra or {})
    meta['imported_from'] = str(src.relative_to(REPO_ROOT))
    meta['run_key'] = dst.name
    meta['arm'] = dst.name.rsplit('_seed', 1)[0]
    with open(dst / 'run_meta.json', 'w') as f:
        json.dump(meta, f, indent=2, default=str)
    return True, 'imported from %s' % src.relative_to(REPO_ROOT)


def try_import_ref(arm_name, seed):
    '''Both references come from ablation_keep_disease_only, which used this exact config.'''
    if not IMPORT_REFS:
        return False, 'disabled'
    a = ARMS_BY_NAME[arm_name]
    expect = {'split': SPLIT, 'seed': seed,
              'n_edge_types_after': a['expect_etypes'],
              'model_config': {k: MODEL_CFG[k] for k in MODEL_KEYS_CHECKED},
              'pretrain': PRETRAIN_CFG, 'finetune': FINETUNE_CFG}
    return import_matching_run(
        KEEP_ROOT / ('%s_seed%d' % (arm_name, seed)), run_dir(arm_name, seed), expect,
        extra={'experiment': 'keep_only_controls',
               'kept_relations_requested': a['keep'],
               'expect_encoder_params': a['expect_encoder_params']})


if IMPORT_REFS:
    for _arm_name in (REF_ARM, REF_ARM_2):
        n_ok = 0
        for s in SEEDS:
            if is_done(_arm_name, s):
                n_ok += 1
                continue
            ok, why = try_import_ref(_arm_name, s)
            n_ok += bool(ok)
            if not ok:
                print('[import] %s_seed%d: will train (%s)' % (_arm_name, s, why))
        print('[import] %-24s %d/%d available' % (_arm_name, n_ok, len(SEEDS)))

# --- timing estimate ------------------------------------------------------------------------------
def _mean_seconds(root, arm):
    ts = []
    for p in sorted(Path(root).glob('%s_seed*/run_meta.json' % arm)):
        r = json.loads(p.read_text())
        if r.get('pretrain_seconds') and r.get('finetune_seconds'):
            ts.append(r['pretrain_seconds'] + r['finetune_seconds'])
    return float(np.mean(ts)) if ts else None


_s9 = _mean_seconds(KEEP_ROOT, 'only_disease_relations') or 180.0
print('\n[timing] a 9-edge-type run averaged %.1f min in ablation_keep_disease_only' % (_s9 / 60))

rows, total_min = [], 0.0
for a in ARMS:
    seeds = arm_seeds(a)
    done = [s for s in seeds if is_done(a['arm'], s)]
    todo = len(seeds) - len(done)
    # the matched arm is ~5.6x the encoder, so assume ~2x wall clock; the rest match the 9-etype runs
    per_min = (_s9 / 60) * (2.0 if a['arm'] == 'only_disease_matched' else 1.0)
    est = 0.0 if a['arm'] in (REF_ARM, REF_ARM_2) else todo * per_min
    total_min += est
    rows.append({'arm': a['arm'], 'edge types': a['expect_etypes'],
                 'encoder params': format(a['expect_encoder_params'], ','),
                 'done': len(done), 'todo': todo, 'est GPU-h': round(est / 60, 2)})
display(pd.DataFrame(rows).style.hide(axis='index'))

RUN_KEYS = [(a['arm'], s) for s in SEEDS for a in ARMS if s in arm_seeds(a)]
pending = [k for k in RUN_KEYS if not is_done(*k)]
print('\n%d run_keys, %d done, %d pending | estimated %.1f GPU-hours remaining'
      % (len(RUN_KEYS), len(RUN_KEYS) - len(pending), len(pending), total_min / 60))
for arm, seed in RUN_KEYS:
    print('  [%s] %s_seed%d' % ('x' if is_done(arm, seed) else ' ', arm, seed))

---
## 5 - The grid

Guards that fire before training, per run:

1. the surviving edge-type count equals the arm's `expect_etypes`;
2. `encoder_params(model)` equals the closed form **exactly** - this is what makes
   `only_disease_matched` a capacity-matched arm rather than a claim;
3. all 10 node types survive with the same indices;
4. the signature probe reports the expected sources - **both** for the matched arm, **exactly one** for
   each padding arm, and the maximum must be non-zero unless the arm is the declared degenerate control.

`run_meta.json` is written last and `is_done` needs both files, so an interrupted run stays pending.
Safe to re-run from cell 1.

In [ ]:
def run_one(arm_name, seed, verbose=True):
    arm  = ARMS_BY_NAME[arm_name]
    rdir = run_dir(arm_name, seed)
    el_csv, meta_json = rdir / 'edge_level_test_metrics.csv', rdir / 'run_meta.json'

    if el_csv.exists() and meta_json.exists():
        if verbose:
            print('[skip] %s_seed%d -> cached' % (arm_name, seed))
        return pd.read_csv(el_csv)

    if arm_name in (REF_ARM, REF_ARM_2):
        ok, why = try_import_ref(arm_name, seed)
        if ok:
            print('[skip] %s_seed%d -> %s' % (arm_name, seed, why))
            return pd.read_csv(el_csv)

    cfg = dict(MODEL_CFG, **arm['overrides'])
    degenerate_ok = arm_name == 'arbitrary9_no_disease'

    print('\n' + '#' * 78)
    print('# %s   seed %d   keep=%s' % (arm_name, seed, arm['keep']))
    print('# width %d/%d/%d   expect %d edge types, %s encoder params'
          % (cfg['n_inp'], cfg['n_hid'], cfg['n_out'], arm['expect_etypes'],
             format(arm['expect_encoder_params'], ',')))
    print('#' * 78)

    base = get_seed_data(seed)

    set_all_seeds(seed)
    d = KeptData(base, arm['keep'], DATA_FOLDER, SPLIT)
    etypes_after = sorted('%s|%s|%s' % et for et in d.G.edge_types)
    n_et = len(d.G.edge_types)
    rel_names = {et[1] for et in d.G.edge_types}

    # ---- guard 1: edge-type count ----
    assert n_et == arm['expect_etypes'], \
        '%s: %d edge types, expected %d -> %s' % (arm_name, n_et, arm['expect_etypes'], etypes_after)

    # ---- guard 3: node types untouched ----
    want_nodes = set(d.df.x_type) | set(d.df.y_type)
    assert set(d.G.node_types) == want_nodes, \
        'node types changed: %s' % (set(d.G.node_types) ^ want_nodes)

    dropped = len(base['df_train']) - len(d.df_train)
    print('[graph] train edges %d of %d (%.2f%%) | %d edge types | %d node types'
          % (len(d.df_train), len(base['df_train']),
             100.0 * len(d.df_train) / len(base['df_train']), n_et, len(d.G.node_types)))

    set_all_seeds(seed)
    m = TxGNN(data=d, weight_bias_track=False, exp_name='%s_seed%d' % (arm_name, seed), device=DEVICE)
    m.model_initialize(**cfg)

    # ---- guard 2: capacity is what the arm claims ----
    enc_p, tot_p = encoder_params(m.model), get_n_params(m.model)
    assert enc_p == arm['expect_encoder_params'], \
        '%s: encoder params %d != expected %d' % (arm_name, enc_p, arm['expect_encoder_params'])
    print('[model] encoder params %s (%.2f%% of the 50-etype baseline) | total %s'
          % (format(enc_p, ','), 100.0 * enc_p / expected_encoder_params(50, 100, 100, 100),
             format(tot_p, ',')))

    t0 = time.time()
    set_all_seeds(seed)
    m.pretrain(**PRETRAIN_CFG)
    pre_s = time.time() - t0

    t0 = time.time()
    set_all_seeds(seed)
    m.finetune(**FINETUNE_CFG)
    fin_s = time.time() - t0

    # ---- guard 4: the signature is what the arm needs it to be ----
    sig_sources = [r for r in SIG_ETYPES if r in rel_names]
    sig_dim, sig_max = signature_probe(m.best_model)
    print('[signature] sources present %s | dim %s | max %s' % (sig_sources, sig_dim, sig_max))
    if arm_name == 'only_disease_matched':
        assert len(sig_sources) == 2, 'matched arm lost a signature source: %s' % sig_sources
    elif arm_name in ('dd_plus_padding', 'dp_plus_padding'):
        assert len(sig_sources) == 1, \
            '%s must keep exactly one signature source, has %s' % (arm_name, sig_sources)
    if not degenerate_ok:
        assert sig_max is None or sig_max > 0, \
            '%s has an all-zero signature - degenerate, see cell 0' % arm_name

    (auroc_rel, auprc_rel, micro_auroc, micro_auprc, macro_auroc, macro_auprc), test_loss = \
        evaluate_fb(m.best_model, m.g_test_pos, m.g_test_neg, m.G, m.dd_etypes, DEVICE, mode='test')

    el = pd.DataFrame(
        [('%s --%s--> %s' % et, auroc_rel.get(et, np.nan), auprc_rel.get(et, np.nan))
         for et in DD_ETYPES],
        columns=['relation', 'AUROC', 'AUPRC'])
    el.loc[len(el)] = ['MICRO (drug-disease)', micro_auroc, micro_auprc]
    el.loc[len(el)] = ['MACRO (drug-disease)',
                       float(np.nanmean([auroc_rel.get(et, np.nan) for et in DD_ETYPES])),
                       float(np.nanmean([auprc_rel.get(et, np.nan) for et in DD_ETYPES]))]
    el.loc[len(el)] = ['MACRO (all relations)', macro_auroc, macro_auprc]

    rdir.mkdir(parents=True, exist_ok=True)
    el.to_csv(el_csv, index=False)

    ind = auprc_rel.get(('drug', 'indication', 'disease'), np.nan)
    if not degenerate_ok and abs(float(test_loss) - np.log(2)) < 1e-5:
        print('[WARN] test BCE == ln 2 exactly - this looks like the degenerate regime of cell 0.')

    meta = {
        'run_key': '%s_seed%d' % (arm_name, seed),
        'arm': arm_name, 'seed': seed, 'split': SPLIT, 'profile': PROFILE,
        'experiment': 'keep_only_controls',
        'note': arm['note'],
        'kept_relations_requested': arm['keep'],
        'kept_relation_names': sorted(rel_names),
        'n_edge_types_after': n_et,
        'edge_types_after': etypes_after,
        'n_node_types': len(d.G.node_types),
        'train_edges_before': len(base['df_train']),
        'train_edges_after': len(d.df_train),
        'train_edges_kept_pct': 100.0 * len(d.df_train) / len(base['df_train']),
        'dropped_edges_total': dropped,
        'signature_disease_etypes': SIG_ETYPES,
        'signature_sources_present': sig_sources,
        'signature_dim': sig_dim, 'signature_max': sig_max,
        'encoder_params': enc_p,
        'expect_encoder_params': arm['expect_encoder_params'],
        'n_params': tot_p,
        'proto': cfg['proto'],
        'model_config': m.config,
        'width': [cfg['n_inp'], cfg['n_hid'], cfg['n_out']],
        'pretrain': PRETRAIN_CFG, 'finetune': FINETUNE_CFG,
        'pretrain_seconds': pre_s, 'finetune_seconds': fin_s,
        'test_bce_loss': float(test_loss),
        'torch': torch.__version__,
        'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'finished_at': time.strftime('%Y-%m-%dT%H:%M:%S'),
    }
    with open(meta_json, 'w') as f:
        json.dump(meta, f, indent=2, default=str)

    if SAVE_MODELS:
        mdir = REPO_ROOT / 'saved_models' / 'ablation_keep_only_controls' / meta['run_key']
        mdir.mkdir(parents=True, exist_ok=True)
        m.save_model(str(mdir))

    con = auprc_rel.get(('drug', 'contraindication', 'disease'), np.nan)
    print('[done] %s_seed%d | %.1f min | indication %.4f | contraindication %.4f -> %s'
          % (arm_name, seed, (pre_s + fin_s) / 60, ind, con, rdir.relative_to(REPO_ROOT)))

    del m, d, auroc_rel, auprc_rel
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return el

In [ ]:
if RUN_GRID:
    todo = [(a['arm'], s) for s in SEEDS for a in ARMS
            if s in arm_seeds(a) and not is_done(a['arm'], s)]
    if STOP_AFTER_N_RUNS:
        todo = todo[:STOP_AFTER_N_RUNS]

    print('%d run(s) pending.' % len(todo))
    grid_t0 = time.time()
    for i, (arm_name, seed) in enumerate(todo, 1):
        print('\n>>> %d/%d  %s_seed%d  (elapsed %.1f h)'
              % (i, len(todo), arm_name, seed, (time.time() - grid_t0) / 3600))
        run_one(arm_name, seed)
    print('\nGrid finished %d run(s) in %.1f h.' % (len(todo), (time.time() - grid_t0) / 3600))
else:
    print('RUN_GRID is False - no training. Set it in section 2.')

---
## 6 - Aggregate

In [ ]:
def _load_all(root):
    recs = []
    for meta_p in sorted(Path(root).glob('*/run_meta.json')):
        el_p = meta_p.parent / 'edge_level_test_metrics.csv'
        if not el_p.exists():
            continue
        meta = json.loads(meta_p.read_text())
        el = pd.read_csv(el_p).set_index('relation')
        for rel, row in el.iterrows():
            recs.append({'arm': meta['arm'], 'seed': meta['seed'], 'relation': rel,
                         'AUROC': row['AUROC'], 'AUPRC': row['AUPRC'],
                         'encoder_params': meta.get('encoder_params'),
                         'n_params': meta.get('n_params'),
                         'n_edge_types': meta.get('n_edge_types_after'),
                         'train_edges': meta.get('train_edges_after', meta.get('train_edges')),
                         'sig_sources': ','.join(meta.get('signature_sources_present') or []),
                         'sig_max': meta.get('signature_max'),
                         'bce': meta.get('test_bce_loss'),
                         'imported': 'imported_from' in meta})
    return pd.DataFrame(recs)


def _ci95(x):
    x = np.asarray(x, dtype=float)
    return 1.96 * x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan


ROW = {'indication': 'drug --indication--> disease',
       'contraindication': 'drug --contraindication--> disease',
       'micro': 'MICRO (drug-disease)'}

long = _load_all(ABL_ROOT)
summary = wide = ref = ref2 = None

if long.empty:
    print('No runs on disk yet - run section 5 first.')
else:
    wide = (long[long.relation.isin(ROW.values())]
            .pivot_table(index=['arm', 'seed'], columns='relation', values='AUPRC')
            .rename(columns={v: k for k, v in ROW.items()}))
    _first = lambda s: s.dropna().iloc[0] if s.notna().any() else np.nan
    info = long.groupby('arm').agg(encoder_params=('encoder_params', _first),
                                   n_params=('n_params', _first),
                                   n_edge_types=('n_edge_types', _first),
                                   train_edges=('train_edges', _first),
                                   sig_sources=('sig_sources', 'first'),
                                   sig_max=('sig_max', _first))
    arms_present = set(wide.index.get_level_values('arm'))
    ref  = wide.xs(REF_ARM,   level='arm') if REF_ARM   in arms_present else None
    ref2 = wide.xs(REF_ARM_2, level='arm') if REF_ARM_2 in arms_present else None
    if ref is None:
        print('[warn] no %s runs - deltas will be NaN.' % REF_ARM)

    out = []
    for arm, g in wide.groupby(level='arm'):
        g = g.droplevel('arm')
        rec = {'arm': arm, 'n_seeds': len(g),
               'edge types': info.loc[arm, 'n_edge_types'],
               'encoder_params': info.loc[arm, 'encoder_params'],
               'sig_sources': info.loc[arm, 'sig_sources'],
               'sig_max': info.loc[arm, 'sig_max']}
        for key in ROW:
            rec['%s_AUPRC' % key] = g[key].mean()
            rec['%s_CI' % key] = _ci95(g[key])
            for tag, r in (('', ref), ('_vs9', ref2)):
                if r is None:
                    continue
                paired = (g[key] - r[key]).dropna()
                rec['%s_delta%s' % (key, tag)] = paired.mean() if len(paired) else np.nan
                rec['%s_delta%s_CI' % (key, tag)] = _ci95(paired)
                rec['%s_hurt%s' % (key, tag)] = int((paired < 0).sum())
                rec['%s_n_paired%s' % (key, tag)] = len(paired)
        out.append(rec)

    order = [a['arm'] for a in ARMS]
    summary = (pd.DataFrame(out)
                 .assign(_o=lambda d: d.arm.map({a: i for i, a in enumerate(order)}))
                 .sort_values('_o').drop(columns='_o').reset_index(drop=True))
    summary.to_csv(ABL_ROOT / 'keep_only_controls_summary.csv', index=False)

    cols = [c for c in ['arm', 'n_seeds', 'edge types', 'encoder_params', 'sig_sources',
                        'indication_AUPRC', 'indication_delta', 'indication_delta_CI',
                        'indication_hurt', 'contraindication_AUPRC', 'contraindication_delta',
                        'contraindication_delta_CI', 'micro_AUPRC', 'micro_delta']
            if c in summary.columns]
    print('%d arms, %d runs -> %s\n'
          % (len(summary), len(wide), (ABL_ROOT / 'keep_only_controls_summary.csv').relative_to(REPO_ROOT)))
    display(summary[cols].style
            .format('{:.4f}', subset=[c for c in cols if c.endswith(('AUPRC', 'delta', 'CI'))], na_rep='-')
            .format({'encoder_params': '{:,.0f}', 'edge types': '{:.0f}'}, na_rep='-')
            .hide(axis='index'))

    degen = summary[(summary.sig_max == 0) | (summary.arm == 'arbitrary9_no_disease')]
    if len(degen):
        print('\n[degenerate] all-zero signature, do not report as a floor: %s'
              % ', '.join(degen.arm))

### 6.1 - The two answers

**Objection 1.** `only_disease_matched` vs `full_kg` is the matched-capacity estimate of what the 41
dropped relations carry. If it is much smaller than -0.066, most of the keep-only loss was capacity, and
the information content of those 41 edge types is smaller still. If it is close to -0.066, capacity was
not the explanation and -0.066 was already a fair estimate.

**Objection 2.** `only_disease_relations` minus each padding arm is the marginal value of the *other*
signature relation **at fixed encoder size and fixed edge-type count** - something leave-one-out from the
full KG cannot measure, because there the encoder shrinks too. Compare these against the published
leave-one-out numbers (`disease_protein` -0.1078, `disease_disease` -0.0438 indication).

In [ ]:
if summary is None or ref is None:
    print('Need finished %s runs plus at least one other arm.' % REF_ARM)
else:
    s = summary.set_index('arm')
    base9 = expected_encoder_params(9, 100, 100, 100)

    print('=' * 78)
    print('OBJECTION 1 - matched capacity')
    print('=' * 78)
    if 'only_disease_matched' in s.index:
        for key in ['indication', 'contraindication', 'micro']:
            d9 = s.loc[REF_ARM_2, '%s_delta' % key] if REF_ARM_2 in s.index else np.nan
            dm = s.loc['only_disease_matched', '%s_delta' % key]
            print('  %-17s 9 etypes @181,800: %+.4f  |  9 etypes @%s: %+.4f  |  capacity share %s'
                  % (key, d9, format(int(base9 * 0) + 1010880, ','), dm,
                     ('%.0f%%' % (100 * (1 - dm / d9))) if (pd.notna(d9) and d9) else '-'))
        print('\n  -> the matched number is the estimate to quote; the 181,800 one is an upper bound.')
    else:
        print('  only_disease_matched has no runs yet.')

    print('\n' + '=' * 78)
    print('OBJECTION 2 - relation identity at fixed capacity (vs only_disease_relations)')
    print('=' * 78)
    LOO = {'dd_plus_padding':  ('disease_protein', -0.1078, -0.1258),
           'dp_plus_padding':  ('disease_disease', -0.0438, -0.0036)}
    rows = []
    for arm, (missing, loo_ind, loo_con) in LOO.items():
        if arm not in s.index or REF_ARM_2 not in s.index:
            continue
        rows.append({
            'arm': arm,
            'relation it is missing': missing,
            'indication vs 9-etype': s.loc[arm, 'indication_delta_vs9'],
            'leave-one-out (full KG)': loo_ind,
            'contraind. vs 9-etype': s.loc[arm, 'contraindication_delta_vs9'],
            'leave-one-out (full KG) ': loo_con,
            'n': int(s.loc[arm, 'n_seeds']),
        })
    if rows:
        display(pd.DataFrame(rows).style.format('{:.4f}', subset=[
            'indication vs 9-etype', 'leave-one-out (full KG)',
            'contraind. vs 9-etype', 'leave-one-out (full KG) '], na_rep='-').hide(axis='index'))
        print('  Agreement between the two columns means the leave-one-out effect was not a capacity')
        print('  artefact. A much smaller "vs 9-etype" effect means it partly was.')
    else:
        print('  padding arms have no runs yet.')

In [ ]:
# Per-seed paired deltas against both references.
if summary is not None:
    for label, r in (('full_kg', ref), ('only_disease_relations', ref2)):
        if r is None:
            continue
        for key in ['indication', 'contraindication']:
            arms = [a for a in summary.arm if a in wide.index.get_level_values('arm') and a != label]
            if not arms:
                continue
            tbl = pd.DataFrame({a: (wide.xs(a, level='arm')[key] - r[key]) for a in arms}).T
            seeds_present = list(tbl.columns)
            tbl.columns = ['seed %d' % x for x in seeds_present]
            tbl['mean'] = tbl.mean(axis=1)
            tbl['hurt'] = (tbl[['seed %d' % x for x in seeds_present]] < 0).sum(axis=1)
            print('\npaired delta AUPRC vs %s - %s  (negative = the arm is worse)' % (label, key))
            display(tbl.style.format('{:.4f}', subset=[c for c in tbl.columns if c != 'hurt'],
                                     na_rep='-'))

---
## 7 - What to write up

Fill these two sentences from section 6.1 and they replace the corresponding hedges in
`docs/ABLATION_FINDINGS.md` section 7 and section 11, and in the comment-paper draft section 1.4:

> At matched encoder capacity (9 edge types, 1,010,880 parameters vs the baseline's 1,010,000),
> reducing the knowledge graph to the task labels plus the two relations the similarity signature reads
> costs **X** indication AUPRC. The 41 dropped edge types are therefore worth **X**, not merely
> "at most 0.066".

> Holding edge-type count and encoder size fixed at 9 and 181,800, keeping only `disease_protein`
> costs **Y** relative to keeping both signature relations, and keeping only `disease_disease` costs
> **Z**. These bracket the leave-one-out estimates obtained from the full graph, so those estimates were
> not capacity artefacts.

### Still not addressed by this notebook

1. **Frozen node features** (`utils.py:1127`, `requires_grad=False`) - unchanged in every arm here.
   See `ablation_learnable_embeddings.ipynb`.
2. **One split protocol and the lenient metric** - `complex_disease`, edge-level AUPRC, one sampled
   negative per positive, chance exactly 0.500. See `ablation_full_ranking.ipynb`.
3. **Pretraining held at one epoch** in every arm. See `ablation_pretrain_and_gate.ipynb`.
4. The padding relations are inert *by construction*, which is the point, but it does mean the padding
   arms spend ~2/9 of their encoder on parameters that cannot affect a prediction. That is exactly what
   makes them capacity-matched to `only_disease_relations`, which spends the same parameters on the same
   kind of unreachable structure for any disease whose only route is a dropped relation.